# Google calendar evets

Here we will cover
```
check_availability
create_event
update_event
list_events
delete_event

# check_availability

In [31]:
import os
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

# SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]
SCOPES = [ "https://www.googleapis.com/auth/calendar"]
creds = None

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json",
            SCOPES
        )
        creds = flow.run_local_server(port=0)

    with open("token.json", "w") as token:
        token.write(creds.to_json())


# Build service
service = build("calendar", "v3", credentials=creds)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=737381595269-a4d6pt3407tq0okofigrvjjntlueaspc.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A54534%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=dhR7qx9AYjU8GDUTUUAYnwFCHv5we6&code_challenge=7Pd3lkDYQ1Rg80myP_bUpHIv8eUKSy9bC_bQiDVxnnY&code_challenge_method=S256&access_type=offline


In [14]:
def check_availability(
    calendar_id: str,
    start_time: str,
    end_time: str
) -> str:
    """
    Check whether a Google Calendar is free during a given time period.

    Args:
        calendar_id: Calendar email address or "primary".
        start_time: ISO-8601 datetime (e.g. "2026-07-11T18:00:00+05:30")
        end_time: ISO-8601 datetime (e.g. "2026-07-11T19:00:00+05:30")

    Returns:
        A message indicating whether the calendar is free or busy.
    """

    body = {
        "timeMin": start_time,
        "timeMax": end_time,
        "items": [
            {"id": calendar_id}
        ]
    }

    result = service.freebusy().query(body=body).execute()

    busy = result["calendars"][calendar_id]["busy"]

    if not busy:
        return "Calendar is FREE."

    output = ["Calendar is BUSY during:"]

    for interval in busy:
        output.append(
            f"{interval['start']}  -->  {interval['end']}"
        )

    return "\n".join(output)

In [15]:
# Am I available on 11 July 2026 from 6 PM to 7 PM, test it like this:
response = check_availability(
    calendar_id="primary",
    start_time="2026-07-11T17:30:00+05:30",
    end_time  ="2026-07-11T18:30:00+05:30"
)

print(response)

Calendar is BUSY during:
2026-07-11T12:30:00Z  -->  2026-07-11T13:00:00Z


In [36]:
# Am I available on 11 July 2026 from 5:30 PM to 6:30 PM, test it like this:
response = check_availability(
    calendar_id="ash322.ash422@gmail.com",
    start_time="2026-07-11T17:30:00+05:30",
    end_time  ="2026-07-11T18:30:00+05:30"
)

print(response)

Calendar is BUSY during:
2026-07-11T12:30:00Z  -->  2026-07-11T13:00:00Z


# create_event()
Following send an invitation to the attendees

In [32]:
def create_event(
    summary: str,
    start_time: str,
    end_time: str,
    attendees: list[str],
    description: str = "",
    location: str = ""
) -> str:
    """
    Create a Google Calendar meeting.

    Args:
        summary: Meeting title.
        start_time: ISO-8601 datetime.
        end_time: ISO-8601 datetime.
        attendees: List of attendee email addresses.
        description: Optional meeting description.
        location: Optional location.

    Returns:
        Details of the created meeting.
    """

    event = {
        "summary": summary,
        "location": location,
        "description": description,
        "start": {
            "dateTime": start_time,
        },
        "end": {
            "dateTime": end_time,
        },
        "attendees": [
            {"email": email} for email in attendees
        ],
    }

    created_event = (
        service.events()
        .insert(
            calendarId="primary",
            body=event,
            sendUpdates="all"      # Sends invitation emails
        )
        .execute()
    )

    return (
        f"Meeting created successfully!\n"
        f"Title: {created_event['summary']}\n"
        f"Starts: {created_event['start']['dateTime']}\n"
        f"Ends: {created_event['end']['dateTime']}\n"
        f"Event ID: {created_event['id']}\n"
        f"Event Link: {created_event['htmlLink']}"
    )

In [40]:
response = create_event(
    summary="GenAI Discussion",
        start_time= "2026-07-15T18:00:00+05:30",
        end_time="2026-07-15T19:00:00+05:30",
        attendees=["ashamlodhiya@gmail.com", ],
        description="Discuss AI Agents"
)

print(response)

Meeting created successfully!
Title: GenAI Discussion
Starts: 2026-07-15T18:00:00+05:30
Ends: 2026-07-15T19:00:00+05:30
Event ID: k475pge23q2iqehdotit8v3m40
Event Link: https://www.google.com/calendar/event?eid=azQ3NXBnZTIzcTJpcWVoZG90aXQ4djNtNDAgYXNoMzIyLmFzaDQyMkBt


In [42]:
# Extract event id
import re


# Match text following 'Event ID: ' up to the end of the line
match = re.search(r"Event ID:\s*([a-z0-9]+)", response)

if match:
    EVENT_ID = match.group(1)
    print(f"Extracted Event ID: {event_id}")
else:
    print("Event ID not found.")

Extracted Event ID: k475pge23q2iqehdotit8v3m40


# update_event()

In [34]:
def update_event(
    event_id: str,
    summary: str,
    start_time: str,
    end_time: str,
    attendees: list[str],
    description: str = "",
    location: str = ""
) -> str:
    """
    Update an existing Google Calendar meeting.

    Args:
        event_id: Google Calendar event ID.
        summary: New meeting title.
        start_time: ISO-8601 datetime.
        end_time: ISO-8601 datetime.
        attendees: List of attendee email addresses.
        description: Optional description.
        location: Optional location.

    Returns:
        Status message.
    """

    event = service.events().get(
        calendarId="primary",
        eventId=event_id
    ).execute()

    event["summary"] = summary
    event["description"] = description
    event["location"] = location

    event["start"] = {
        "dateTime": start_time
    }

    event["end"] = {
        "dateTime": end_time
    }

    event["attendees"] = [
        {"email": email}
        for email in attendees
    ]

    updated = (
        service.events()
        .update(
            calendarId="primary",
            eventId=event_id,
            body=event,
            sendUpdates="all"
        )
        .execute()
    )

    return (
        f"Meeting updated successfully.\n"
        f"Title: {updated['summary']}\n"
        f"Starts: {updated['start']['dateTime']}\n"
        f"Ends: {updated['end']['dateTime']}"
    )

In [43]:
response = update_event(
    event_id= EVENT_ID, # retrieved from above
    summary= "Updated GenAI Class",
    start_time= "2026-07-15T19:00:00+05:30",
    end_time= "2026-07-15T20:00:00+05:30",
    attendees= ["john@gmail.com"],
    description= "Updated discussion"
)

print(response)

Meeting updated successfully.
Title: Updated GenAI Class
Starts: 2026-07-15T19:00:00+05:30
Ends: 2026-07-15T20:00:00+05:30


## List all upcoming events

In [45]:

from datetime import datetime, timezone
from datetime import datetime, timezone


def list_upcoming_events(service, max_results=10):
    """Fetches and prints upcoming events from the primary calendar."""
    # Get current time in UTC ISO format
    now = datetime.now(timezone.utc).isoformat()

    # Call the Calendar API
    events_result = (
        service.events()
        .list(
            calendarId="primary",
            timeMin=now,
            maxResults=max_results,
            singleEvents=True,
            orderBy="startTime",
        )
        .execute()
    )

    events = events_result.get("items", [])

    if not events:
        print("No upcoming events found.")
        return []

    # Print and return the events
    for event in events:
        print("Title :", event.get("summary", "No Title"))
        print("ID    :", event.get("id"))
        print("Start :", event["start"].get("dateTime", event["start"].get("date")))
        print("-" * 40)

    return events


events = list_upcoming_events(service, max_results=10)
print(events)

Title : GenAI Class
ID    : 54ajf2ihfooqpb7q6o7heag511
Start : 2026-07-11T18:00:00+05:30
----------------------------------------
Title : Updated GenAI Class
ID    : k475pge23q2iqehdotit8v3m40
Start : 2026-07-15T19:00:00+05:30
----------------------------------------
Title : Updated GenAI Class
ID    : nbneac7d3q0rb2vbuge1idok88
Start : 2026-07-15T19:00:00+05:30
----------------------------------------


[{'kind': 'calendar#event',
  'etag': '"3567332557931870"',
  'id': '54ajf2ihfooqpb7q6o7heag511',
  'status': 'confirmed',
  'htmlLink': 'https://www.google.com/calendar/event?eid=NTRhamYyaWhmb29xcGI3cTZvN2hlYWc1MTEgYXNoMzIyLmFzaDQyMkBt',
  'created': '2026-07-10T06:50:58.000Z',
  'updated': '2026-07-10T06:51:18.965Z',
  'summary': 'GenAI Class',
  'creator': {'email': 'ash322.ash422@gmail.com', 'self': True},
  'organizer': {'email': 'ash322.ash422@gmail.com', 'self': True},
  'start': {'dateTime': '2026-07-11T18:00:00+05:30',
   'timeZone': 'Asia/Kolkata'},
  'end': {'dateTime': '2026-07-11T21:00:00+05:30', 'timeZone': 'Asia/Kolkata'},
  'iCalUID': '54ajf2ihfooqpb7q6o7heag511@google.com',
  'sequence': 0,
  'reminders': {'useDefault': True},
  'eventType': 'default'},
 {'kind': 'calendar#event',
  'etag': '"3567362776102526"',
  'id': 'k475pge23q2iqehdotit8v3m40',
  'status': 'confirmed',
  'htmlLink': 'https://www.google.com/calendar/event?eid=azQ3NXBnZTIzcTJpcWVoZG90aXQ4djNtNDAgYXN

# delete_event()

In [48]:
def delete_event(event_id: str) -> str:
    """
    Delete a Google Calendar event.

    Args:
        event_id: Google Calendar event ID.

    Returns:
        Status message.
    """

    service.events().delete(
        calendarId="primary",
        eventId=event_id,
        sendUpdates="all"
    ).execute()

    return "Meeting deleted successfully."

In [50]:
response = delete_event(event_id = EVENT_ID)

print(response)

Meeting deleted successfully.
